# Tutorial 06 — Contribution Analysis & Interpretation

Companion explainer: **06_contribution_analysis.md**. From a single score to
*why*: the two fundamental cuts (by process, by flow), bw2analyzer
conveniences, recursive supply-chain traversal, stage grouping, and plots.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import bw2data as bd
import bw2calc as bc
import bw2analyzer as ba

bd.projects.set_current("bw25-tutorials")
BIOSPHERE = next(d for d in bd.databases if "biosphere" in d.lower())
bio = bd.Database(BIOSPHERE)

def find_flow(name, categories=("air",)):
    return next(f for f in bio if f["name"] == name and f["categories"] == categories)

co2, ch4 = find_flow("Carbon dioxide, fossil"), find_flow("Methane, fossil")

## Rebuild kettle with stage tags on foreground activities

In [2]:
DB = "t06_kettle"
if DB in bd.databases:
    del bd.databases[DB]
bd.Database(DB).write({
    (DB, "elec"): {"name": "electricity, coal", "unit": "kilowatt hour",
                   "stage": "energy", "exchanges": [
        {"input": (DB, "elec"), "amount": 1.0, "type": "production"},
        {"input": co2.key, "amount": 0.95, "type": "biosphere"},
        {"input": ch4.key, "amount": 0.0002, "type": "biosphere"}]},
    (DB, "steel"): {"name": "steel", "unit": "kilogram",
                    "stage": "materials", "exchanges": [
        {"input": (DB, "steel"), "amount": 1.0, "type": "production"},
        {"input": (DB, "elec"), "amount": 2.9, "type": "technosphere"},
        {"input": co2.key, "amount": 1.9, "type": "biosphere"}]},
    (DB, "pp"): {"name": "polypropylene", "unit": "kilogram",
                 "stage": "materials", "exchanges": [
        {"input": (DB, "pp"), "amount": 1.0, "type": "production"},
        {"input": (DB, "elec"), "amount": 2.0, "type": "technosphere"},
        {"input": co2.key, "amount": 1.6, "type": "biosphere"}]},
    (DB, "kettle"): {"name": "kettle assembly", "unit": "unit",
                     "stage": "assembly", "exchanges": [
        {"input": (DB, "kettle"), "amount": 1.0, "type": "production"},
        {"input": (DB, "steel"), "amount": 1.2, "type": "technosphere"},
        {"input": (DB, "pp"), "amount": 0.4, "type": "technosphere"},
        {"input": (DB, "elec"), "amount": 0.8, "type": "technosphere"}]},
})
kettle = bd.get_node(database=DB, code="kettle")
gwp = next(m for m in bd.methods
           if "IPCC 2013" in str(m) and "GWP100" in str(m).replace(" ", "")
           and "no LT" not in str(m) and "SLCF" not in str(m))
lca = bc.LCA({kettle: 1}, method=gwp)
lca.lci(); lca.lcia()
print("total GWP:", round(lca.score, 4))

13:17:04-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

info     

] 

Vacuuming database            

total GWP:

7.7762

## The two fundamental cuts (manual, from characterized_inventory)

In [3]:
ci = lca.characterized_inventory
by_process = np.asarray(ci.sum(axis=0)).ravel()  # columns
by_flow = np.asarray(ci.sum(axis=1)).ravel()     # rows

proc_df = pd.DataFrame([
    {"process": bd.get_activity(k)["name"], "score": by_process[v],
     "share": by_process[v] / lca.score}
    for k, v in lca.dicts.activity.items()
]).sort_values("score", ascending=False)
print("BY PROCESS (direct/gate-to-gate):")
print(proc_df.to_string(index=False))

flow_df = pd.DataFrame([
    {"flow": bd.get_activity(k)["name"], "score": by_flow[v],
     "share": by_flow[v] / lca.score}
    for k, v in lca.dicts.biosphere.items() if abs(by_flow[v]) > 1e-12
]).sort_values("score", ascending=False)
print("\nBY ELEMENTARY FLOW:")
print(flow_df.to_string(index=False))

BY PROCESS (direct/gate-to-gate):

          process    score    share
electricity, coal 4.856175 0.624494
            steel 2.280000 0.293203
    polypropylene 0.640000 0.082303
  kettle assembly 0.000000 0.000000


BY ELEMENTARY FLOW:

                  flow    score   share
Carbon dioxide, fossil 7.746000 0.99612
       Methane, fossil 0.030175 0.00388

## bw2analyzer conveniences (same info, pre-annotated)

In [4]:
ca = ba.ContributionAnalysis()
print("top processes:")
for s, sup, act in ca.annotated_top_processes(lca, limit=5):
    print(f"   {s: .4f}  ({sup:.3g} units)  {act}")
print("top emissions:")
for s, amt, flow in ca.annotated_top_emissions(lca, limit=5):
    print(f"   {s: .4f}  ({amt:.3g} kg)  {flow}")

top processes:

    4.8562  (5.08 units)  'electricity, coal' (kilowatt hour, None, None)

    2.2800  (1.2 units)  'steel' (kilogram, None, None)

    0.6400  (0.4 units)  'polypropylene' (kilogram, None, None)

    0.0000  (1 units)  'kettle assembly' (unit, None, None)

top emissions:

    7.7460  (7.75 kg)  'Carbon dioxide, fossil' (kilogram, None, ('air',))

    0.0302  (0.00102 kg)  'Methane, fossil' (kilogram, None, ('air',))

## Cumulative supply-chain traversal (direct != cumulative)

In [5]:
import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    ba.print_recursive_calculation(kettle, gwp, amount=1, max_level=4, cutoff=0.02)
print(buf.getvalue())

Fraction of score | Absolute score | Amount | Activity
0001 | 7.776 |     1 | 'kettle assembly' (unit, None, None)
  0.721 | 5.607 |   1.2 | 'steel' (kilogram, None, None)
    0.428 | 3.327 |  3.48 | 'electricity, coal' (kilowatt hour, None, None)
  0.181 | 1.405 |   0.4 | 'polypropylene' (kilogram, None, None)
    0.0983 | 0.7648 |   0.8 | 'electricity, coal' (kilowatt hour, None, None)
  0.0983 | 0.7648 |   0.8 | 'electricity, coal' (kilowatt hour, None, None)


## Grouping by life-cycle stage
Aggregate by-process contributions over the `stage` tag.

In [6]:
stage_scores = {}
for k, v in lca.dicts.activity.items():
    act = bd.get_activity(k)
    stage = act.get("stage", "other")
    stage_scores[stage] = stage_scores.get(stage, 0.0) + by_process[v]
stage_df = pd.Series(stage_scores).sort_values(ascending=False)
print("GWP by stage:")
print(stage_df)

GWP by stage:

energy       4.856175
materials    2.920000
assembly     0.000000
dtype: float64

## Plots: top-process bar + stage stacked bar

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
proc_df.plot.barh(x="process", y="score", ax=axes[0], legend=False, color="#4C72B0")
axes[0].set_title("Direct GWP by process"); axes[0].set_xlabel("kg CO2-eq")
stage_df.plot.bar(ax=axes[1], color="#55A868")
axes[1].set_title("GWP by life-cycle stage"); axes[1].set_ylabel("kg CO2-eq")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
out = "tutorials_outputs_06.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
print("saved", out)
plt.show()

saved

tutorials_outputs_06.png

C:\Users\derne\AppData\Local\Temp\ipykernel_34184\3573103798.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Next: **07 — uncertainty & Monte Carlo**.